# 05: 大规模自动化 RAGAS 评估验证
本脚本现已支持使用外部化的大批量用户查询及真值响应，内置了对两代架构（基准 RAG 和 抗噪 RAG）的自动化交叉推理与直接对比。请确保运行时同级目录下存有 `qa_dataset.csv`。

In [3]:
import os
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from openai import OpenAI
from ragas.llms import llm_factory
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib

# 设置中文字体
matplotlib.rcParams['font.sans-serif'] = ['SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False

import sys
if "." not in sys.path:
    sys.path.append(".")

from importlib import import_module
# 导入封装好的两个 RAG 管道以及 LLM 生成器
NoiseRobustRAG = import_module("03_advanced_rag_pipeline").NoiseRobustRAG
BaselineRAG = import_module("03_advanced_rag_pipeline").BaselineRAG
RAGGenerator = import_module("04_llm_generation").RAGGenerator


C:\Users\24201\AppData\Local\Temp\ipykernel_31776\2085343157.py:5: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
C:\Users\24201\AppData\Local\Temp\ipykernel_31776\2085343157.py:5: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
C:\Users\24201\AppData\Local\Temp\ipykernel_31776\2085343157.py:5: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections impor

In [4]:
API_KEY = "sk-44d6e7d7fee347eeabbc89410989c8c8"  
BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"
MODEL_NAME = "deepseek-v3.2"

print("正在初始化 RAGAS 裁判模型...")
openai_client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
judge_llm = llm_factory(model=MODEL_NAME, client=openai_client)

print("正在加载本地 RAGAS Embedding 模型...")
lc_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-zh-v1.5", 
    model_kwargs={'device': 'cpu'}
)
judge_embeddings = LangchainEmbeddingsWrapper(lc_embeddings)


正在初始化 RAGAS 裁判模型...
正在加载本地 RAGAS Embedding 模型...


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\24201\AppData\Local\Temp\ipykernel_31776\4028599962.py:14: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_embeddings = LangchainEmbeddingsWrapper(lc_embeddings)


In [5]:
print("正在初始化双通道 RAG 管道...")
corpus_path = "./data/processed/tablet_corpus.csv"
faiss_path = "./indices/faiss_index.bin"
bm25_path = "./indices/bm25_index.pkl"

baseline_rag = BaselineRAG(corpus_path, faiss_path)
robust_rag = NoiseRobustRAG(corpus_path, bm25_path, faiss_path)
generator = RAGGenerator(api_key=API_KEY, base_url=BASE_URL, model_name=MODEL_NAME)


正在初始化双通道 RAG 管道...
正在初始化基准 RAG 管道 (FAISS)...


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ 基准模型与索引加载完毕！

正在初始化抗噪 RAG 管道...
使用计算设备: cpu


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ 模型与索引加载完毕！



In [5]:
import logging
import jieba
import pandas as pd # 确保导入了 pandas
# 屏蔽 jieba 烦人的初始化日志
jieba.setLogLevel(logging.INFO)
# ================= 新增这行读取数据集代码 =================
qa_df = pd.read_csv("qa_dataset.csv")
# ==========================================================
# 初始化存储结果的列表
baseline_results = []
robust_results = []
print("\n🚀 开始批量推理 1/2: 基准 RAG (Naive RAG)...")
# 给进度条加上明确的 desc 描述
for idx, row in tqdm(qa_df.iterrows(), total=len(qa_df), desc="Baseline RAG"):
    query, ref = row['user_input'], row['reference']
    base_contexts = baseline_rag.retrieve_context(query)
    base_answer = generator.generate_answer(query, base_contexts)
    baseline_results.append({
        "user_input": query,
        "response": base_answer,
        "retrieved_contexts": [txt for txt, _ in base_contexts],
        "reference": ref
    })

print("\n🚀 开始批量推理 2/2: 抗噪 RAG (Noise-Robust RAG)...")
for idx, row in tqdm(qa_df.iterrows(), total=len(qa_df), desc="Robust RAG"):
    query, ref = row['user_input'], row['reference']
    robust_contexts = robust_rag.retrieve_context(query)
    robust_answer = generator.generate_answer(query, robust_contexts)
    robust_results.append({
        "user_input": query,
        "response": robust_answer,
        "retrieved_contexts": [txt for txt, _ in robust_contexts],
        "reference": ref
    })

# ================== 以下为你需要新增的落盘保存代码 ==================
print("\n✅ 推理阶段完成！正在将双通道生成结果先行落盘保存...")
# 将生成的两个结果列表缓存，立即转换成 DataFrame 并保存为中间 CSV 文件
pd.DataFrame(baseline_results).to_csv("baseline_generation_only.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(robust_results).to_csv("robust_generation_only.csv", index=False, encoding="utf-8-sig")
print("✅ 落盘完毕！文件已永久保存为 baseline_generation_only.csv 和 robust_generation_only.csv")


🚀 开始批量推理 1/2: 基准 RAG (Naive RAG)...


Baseline RAG: 100%|██████████| 200/200 [19:57<00:00,  5.99s/it]



🚀 开始批量推理 2/2: 抗噪 RAG (Noise-Robust RAG)...


Robust RAG: 100%|██████████| 200/200 [22:50<00:00,  6.85s/it]


✅ 推理阶段完成！正在将双通道生成结果先行落盘保存...
✅ 落盘完毕！文件已永久保存为 baseline_generation_only.csv 和 robust_generation_only.csv


In [6]:
# ==========================================
# 断点续传单元格：从本地硬盘加载已生成的 QA 结果，防止重跑浪费 token
# ==========================================
import pandas as pd
import ast
print("正在从本地读取中间生成文件...")
base_file = "baseline_generation_only.csv"
rob_file = "robust_generation_only.csv"
# 从硬盘读取刚刚保存的 CSV
df_base_loaded = pd.read_csv(base_file)
df_rob_loaded = pd.read_csv(rob_file)
# ⚠️ 关键步骤：存入 CSV 后，列表格式会被拉平成字符串结构（如 "['文本1', '文本2']"）
# 我们需要用 ast.literal_eval 将它安全地还原回真正的 Python List 格式，否则 RAGAS 框架会报错
df_base_loaded['retrieved_contexts'] = df_base_loaded['retrieved_contexts'].apply(ast.literal_eval)
df_rob_loaded['retrieved_contexts'] = df_rob_loaded['retrieved_contexts'].apply(ast.literal_eval)
# 将 DataFrame 重新转回字典 List 格式，完全还原你之前的缓存变量
baseline_results = df_base_loaded.to_dict('records')
robust_results = df_rob_loaded.to_dict('records')
print(f"✅ 断点数据加载成功！")
print(f" -> 已恢复 基准 RAG 数据：{len(baseline_results)} 条")
print(f" -> 已恢复 抗噪 RAG 数据：{len(robust_results)} 条")
print("现在您可以直接往下运行 RAGAS 评估单元格了。")

正在从本地读取中间生成文件...
✅ 断点数据加载成功！
 -> 已恢复 基准 RAG 数据：200 条
 -> 已恢复 抗噪 RAG 数据：200 条
现在您可以直接往下运行 RAGAS 评估单元格了。


In [10]:
# ===== 逐条串行评估（避免并发限流）=====
import asyncio
import time
from ragas.dataset_schema import SingleTurnSample

TEST_SIZE = 20  # 先测试 20 条，跑通后改为 len(baseline_results)

metrics_list = [faithfulness, answer_relevancy, context_precision]
metric_names = ["faithfulness", "answer_relevancy", "context_precision"]

async def evaluate_single_sample(sample_dict, metrics, llm, embeddings):
    """对单条样本逐个指标评分"""
    sample = SingleTurnSample(
        user_input=sample_dict["user_input"],
        response=sample_dict["response"],
        retrieved_contexts=sample_dict["retrieved_contexts"],
        reference=sample_dict["reference"],
    )
    scores = {}
    for metric in metrics:
        metric.llm = llm
        metric.embeddings = embeddings
        try:
            score = await metric.single_turn_ascore(sample)
            scores[metric.name] = score
        except Exception as e:
            print(f"  ⚠️ {metric.name} 评估出错: {e}")
            scores[metric.name] = float("nan")
    return scores

async def run_serial_evaluation(results, label, metrics, llm, embeddings, n):
    """串行逐条评估，每条之间间隔 1 秒防限流"""
    all_scores = []
    for i in range(n):
        print(f"\r  [{label}] 评估进度: {i+1}/{n}", end="", flush=True)
        scores = await evaluate_single_sample(results[i], metrics, llm, embeddings)
        all_scores.append(scores)
        time.sleep(1)  # 间隔 1 秒，防止触发速率限制
    print(f"\r  [{label}] 评估进度: {n}/{n} ✅ 完成!")
    return all_scores

# ===== 开始评估 =====
print(f"\n{'='*50}")
print(f"🚀 开始串行 RAGAS 评估（测试模式：前 {TEST_SIZE} 条）\n")

print("📊 评估 [基准 RAG]...")
baseline_scores = asyncio.get_event_loop().run_until_complete(
    run_serial_evaluation(baseline_results, "基准RAG", metrics_list, judge_llm, judge_embeddings, TEST_SIZE)
)

print("\n📊 评估 [抗噪 RAG]...")
robust_scores = asyncio.get_event_loop().run_until_complete(
    run_serial_evaluation(robust_results, "抗噪RAG", metrics_list, judge_llm, judge_embeddings, TEST_SIZE)
)

# 转成 DataFrame 并保存
df_base = pd.DataFrame(baseline_scores)
df_rob = pd.DataFrame(robust_scores)

df_base.to_csv("baseline_ragas_results.csv", index=False, encoding='utf-8-sig')
df_rob.to_csv("robust_ragas_results.csv", index=False, encoding='utf-8-sig')

print(f"\n✅ 评估完成！结果已保存")
print(f"\n📊 【核心指标均分对比】:")
for m in metric_names:
    print(f"  {m:<20}: Baseline = {df_base[m].mean():.4f} | Robust = {df_rob[m].mean():.4f}")



🚀 开始串行 RAGAS 评估（测试模式：前 20 条）

📊 评估 [基准 RAG]...


RuntimeError: This event loop is already running

In [ ]:
# ==========================================
# 数据可视化：指标总体对比图
# 非常适合直接放入论文 “第四章实证分析与讨论” 中
# ==========================================
metrics = ['faithfulness', 'answer_relevancy', 'context_precision']

base_means = [df_base[m].mean() for m in metrics]
rob_means = [df_rob[m].mean() for m in metrics]

print("\n📊 【核心指标均分对比】:")
for idx, m in enumerate(metrics):
    print(f" - {m.capitalize():<18}: Baseline = {base_means[idx]:.4f} | Robust = {rob_means[idx]:.4f}")

# 开始绘制双柱状图
x = range(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
rects1 = ax.bar([i - width/2 for i in x], base_means, width, label='Baseline RAG (基准)', color='#aec7e8')
rects2 = ax.bar([i + width/2 for i in x], rob_means, width, label='Noise-Robust RAG (抗噪)', color='#1f77b4')

ax.set_ylabel('Scores (0 - 1)', fontsize=12)
ax.set_title('基准 RAG 与抗噪 RAG RAGAS 核心评价指标对比', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([m.capitalize() for m in metrics], fontsize=11)
ax.legend()
ax.set_ylim(0, 1.1)

# 为柱子添加具体数字标签
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=10)

autolabel(rects1)
autolabel(rects2)

plt.tight_layout()
plt.show()
